# What CLIP actually is

CLIP (Contrastive Language-Image Pre-training) is two separate encoders trained together:

An image encoder (a ViT or ResNet) that turns an image into a vector
A text encoder (a Transformer) that turns text into a vector

They're trained on ~400M (image, caption) pairs scraped from the internet, with a contrastive loss: for each batch, the correct image-caption pairs should have high cosine similarity, and every mismatched pair in that batch should have low similarity. Over millions of examples, this forces both encoders to project into a shared embedding space — even though the two encoders have completely different architectures internally, their outputs live in the same coordinate system.

That's the property you actually care about for MedRAG: once you have an image embedding and a text embedding in the same space, cosine similarity between them is meaningful. A text query like "chest X-ray showing pneumonia" can be compared directly against a database of image vectors — no separate "caption this image" step needed.

One thing to be upfront about, since it matters for a medical RAG system: CLIP was trained on general internet images, not medical imaging. It'll do fine on your WHO images (mostly diagrams, charts, algorithm flowcharts, not radiology scans), but if you ever added actual clinical images (X-rays, histology slides), general CLIP would be a weak choice — there are medical-domain variants (BiomedCLIP, PMC-CLIP) for that. Worth a one-line note in your report; not a blocker for what you're doing now.

# Imports

In [3]:
import torch
import open_clip
from PIL import Image

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# list available CLIP pretrained checkpoints for ViT-B-32
print(open_clip.list_pretrained()[:5])

torch: 2.3.1+cpu
CUDA available: False
Using device: cpu
[('RN50', 'openai'), ('RN50', 'yfcc15m'), ('RN50', 'cc12m'), ('RN50-quickgelu', 'openai'), ('RN50-quickgelu', 'yfcc15m')]


# Downloads the pretrained CLIP weights

In [4]:
import open_clip

model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.eval()
model.to(device)

print("Model loaded. Output dim:", model.visual.output_dim)

100%|███████████████████████████████████████| 354M/354M [03:14<00:00, 1.82MiB/s]


Model loaded. Output dim: 512


# Load one real WHO image

In [9]:
from PIL import Image
import torch

# pick one real WHO image from your extracted set
img_path = "../data/images/who/anemia_in_pregnancy_page0_img0.png"  # replace with a real path
image = Image.open(img_path).convert("RGB")

image_input = preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    image_features = model.encode_image(image_input)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)  # normalize

print("Vector shape:", image_features.shape)
print("Vector dtype:", image_features.dtype)
print("Norm (should be ~1.0 after normalization):", image_features.norm().item())
print("First 5 values:", image_features[0][:5])

Vector shape: torch.Size([1, 512])
Vector dtype: torch.float32
Norm (should be ~1.0 after normalization): 1.0
First 5 values: tensor([-0.0416, -0.0017, -0.0138,  0.0156,  0.0467])


# Sanity check

In [10]:
texts = [
    "a diagram showing cardiovascular disease risk levels",
    "a photo of a cat",
    "anemia in pregnancy risk factors",
]

text_tokens = tokenizer(texts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

similarities = (image_features @ text_features.T).squeeze(0)

for text, sim in zip(texts, similarities):
    print(f"{sim.item():.4f}  —  {text}")

0.1916  —  a diagram showing cardiovascular disease risk levels
0.1464  —  a photo of a cat
0.1953  —  anemia in pregnancy risk factors
